In [8]:
import numpy as np
# pip install "cvxpy[SCIP]"
import cvxpy as cp
print(cp.installed_solvers())

['CLARABEL', 'OSQP', 'SCIP', 'SCIPY', 'SCS']


In [4]:
def node_to_edge(dv, a):
    """
    converts degree distribution from node perspective to distribution from
    edge perspective, as required in density evolution for example
    """
    max_dv = np.max(dv)
    L_poly = np.zeros(shape=(max_dv+1))
    L_poly[max_dv - dv + 1] = a

    poly = np.polyder(L_poly)
    poly = poly / np.polyval(poly, 1)
    return poly

def get_rate_LDPC(dv, a_dv, dc, a_dc):
    """
    degree distribution from node perspective
    """
    vn_poly = node_to_edge(dv, a_dv)
    cn_poly = node_to_edge(dc, a_dc)

    return 1 - np.polyval(np.polyint(cn_poly), 1) / np.polyval(np.polyint(vn_poly), 1)

def degdist_to_integer(dv, a_dv, dc, a_dc, N):
    rate = get_rate_LDPC(dv, a_dv, dc, a_dc)
    M = int(round(N*(1 - rate)))

    max_dc = int(np.max(dc))
    max_dv = int(np.max(dv))

    # variables
    num_dc_var_slack = cp.Variable(max_dc)
    num_dv_var_slack = cp.Variable(max_dv)
    num_dc_var = cp.Variable(max_dc, integer=True)
    num_dv_var = cp.Variable(max_dv, integer=True)

    a_dv_temp = np.zeros(shape=max_dv)
    a_dv_temp[dv-1] = a_dv
    dv_temp = np.arange(1, max_dv+1)

    a_dc_temp = np.zeros(shape=max_dc)
    a_dc_temp[dc-1] = a_dc
    dc_temp = np.arange(1, max_dc+1)

    # cost function
    cost = cp.Minimize(cp.sum(num_dv_var_slack) + cp.sum(num_dc_var_slack))

    # constraints
    constraints = []
    constraints += [num_dv_var >= 0, num_dv_var <= N] #upper and lower bound for num_dv_var
    constraints += [num_dc_var >= 0, num_dc_var <= M] #upper and lower bound for num_dc_var
    constraints += [num_dv_var_slack >= 0] #lower bound for num_dv_var_slack
    constraints += [num_dc_var_slack >= 0] #lower bound for num_dc_var_slack
    constraints += [cp.sum(num_dv_var) == N] #sumdv
    constraints += [cp.sum(num_dc_var) == M] #sumdc
    constraints += [dv_temp @ num_dv_var - dc_temp @ num_dc_var == 0] #edges
    constraints += [num_dv_var[1] <= a_dv_temp[1] * N] #nodv2inc
    constraints += [num_dv_var[0] == 0] #nodv1
    constraints += [num_dc_var[0] == 0] #nodc1
    constraints += [num_dc_var[1] == 0] #nodc2
    constraints += [num_dv_var/N - a_dv_temp <= num_dv_var_slack] #slackv1
    constraints += [num_dv_var/N - a_dv_temp >= -num_dv_var_slack] #slackv2
    constraints += [num_dc_var/M - a_dc_temp <= num_dc_var_slack] #slackc1
    constraints += [num_dc_var/M - a_dc_temp >= -num_dc_var_slack] #slackc2

    # solver
    problem = cp.Problem(cost, constraints)
    problem.solve(solver=cp.SCIP)

    if num_dv_var.value is None:
        assert False, "Could not solve optimization problem for finding integer degree distribution"
    
    num_dv = np.round(num_dv_var.value).astype(int)
    new_dv = dv_temp[num_dv != 0]
    num_dv = num_dv[num_dv != 0]

    num_dc = np.round(num_dc_var.value).astype(int)
    new_dc = dc_temp[num_dc != 0]
    num_dc = num_dc[num_dc != 0]

    return new_dv, num_dv, new_dc, num_dc